# RGCA Kaggle Baseline Notebook - No GCloud

This is the simplest stable notebook for running the RGCA baseline experiments on Kaggle without `gcloud`.

It does not use Google Cloud authentication. It works from one of these data sources:

1. an existing subset at `/kaggle/working/rgca_pilot_500/data/mimic_subset.jsonl`
2. a private Kaggle input containing `mimic_subset.jsonl`
3. already-local PhysioNet files under `/kaggle/working/physionet`
4. the repo demo dataset, only as a code smoke test

Recommended path right now: use the existing `/kaggle/working/rgca_pilot_500/data/mimic_subset.jsonl` and skip all download/build steps.


## 1. Setup

Run this first. It sets paths and helper functions. It should finish in seconds.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

IS_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/RGCA') if IS_KAGGLE else (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROJECT_ROOT exists:', PROJECT_ROOT.exists())
print('Current directory:', Path.cwd())

def run_command(command, cwd=PROJECT_ROOT):
    print('+', ' '.join(str(part) for part in command))
    return subprocess.run(command, cwd=str(cwd), check=True)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


## 2. Update Repo

Run this in Kaggle to pull the latest code. If internet is off, skip this if `/kaggle/working/RGCA` already exists.


In [ ]:
if IS_KAGGLE:
    if not PROJECT_ROOT.exists():
        run_command(['git', 'clone', 'https://github.com/pidoxy/RGCA.git', str(PROJECT_ROOT)], cwd=Path('/kaggle/working'))
    else:
        run_command(['git', 'pull', 'origin', 'main'])
    run_command([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)])
else:
    print('Local mode: skipping Kaggle repo update.')


## 3. Find Or Choose The Subset

This notebook prefers an existing `mimic_subset.jsonl`. That is the safest path because it avoids re-downloading large data.


In [ ]:
from rgca_baseline.pipeline import load_studies
from rgca_baseline.io_utils import read_jsonl

CANDIDATE_SUBSETS = []
if IS_KAGGLE:
    CANDIDATE_SUBSETS.extend([
        Path('/kaggle/working/rgca_pilot_500/data/mimic_subset.jsonl'),
        Path('/kaggle/working/rgca_private_dataset/mimic_subset.jsonl'),
    ])
    if Path('/kaggle/input').exists():
        CANDIDATE_SUBSETS.extend(sorted(Path('/kaggle/input').rglob('mimic_subset.jsonl')))
else:
    CANDIDATE_SUBSETS.append(PROJECT_ROOT / 'data' / 'demo' / 'demo_studies.jsonl')

print('Candidate subsets:')
for path in CANDIDATE_SUBSETS:
    print('-', path, '| exists:', path.exists())

SUBSET_PATH = next((path for path in CANDIDATE_SUBSETS if path.exists()), None)
if SUBSET_PATH is None:
    print('
No subset found yet. If you have raw PhysioNet files in /kaggle/working/physionet, run Section 4. Otherwise attach/upload a private Kaggle dataset containing mimic_subset.jsonl.')
else:
    print('
Using subset:', SUBSET_PATH)


## 4. Optional: Build Subset From Existing Local PhysioNet Files

Run this only if Section 3 did not find a subset and you already have the metadata, reports, and image folder locally in Kaggle.

No GCloud is used here.


In [ ]:
def find_first(root, name):
    root = Path(root)
    if not root.exists():
        return None
    matches = sorted(root.rglob(name))
    return matches[0] if matches else None

def find_files_dir(root, dataset_name):
    root = Path(root)
    candidates = [
        root / dataset_name / 'files',
        root / dataset_name / '2.1.0' / 'files',
        root / dataset_name / '2.0.0' / 'files',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = sorted(path / 'files' for path in root.rglob(dataset_name) if (path / 'files').exists()) if root.exists() else []
    return matches[0] if matches else None

PHYSIONET_ROOT = Path('/kaggle/working/physionet') if IS_KAGGLE else PROJECT_ROOT / 'data' / 'raw'
METADATA_PATH = find_first(PHYSIONET_ROOT, 'mimic-cxr-2.0.0-metadata.csv.gz')
SPLIT_PATH = find_first(PHYSIONET_ROOT, 'mimic-cxr-2.0.0-split.csv.gz')
LABELS_PATH = find_first(PHYSIONET_ROOT, 'mimic-cxr-2.0.0-chexpert.csv.gz')
MIMIC_REPORTS_ROOT = find_files_dir(PHYSIONET_ROOT, 'mimic-cxr')
MIMIC_JPG_ROOT = find_files_dir(PHYSIONET_ROOT, 'mimic-cxr-jpg')

for name, path in {
    'PHYSIONET_ROOT': PHYSIONET_ROOT,
    'METADATA_PATH': METADATA_PATH,
    'SPLIT_PATH': SPLIT_PATH,
    'LABELS_PATH': LABELS_PATH,
    'MIMIC_REPORTS_ROOT': MIMIC_REPORTS_ROOT,
    'MIMIC_JPG_ROOT': MIMIC_JPG_ROOT,
}.items():
    print(name, '=>', path, '| exists:', bool(path and Path(path).exists()))


In [ ]:
# Build a 500-study pilot subset only if no subset was found.
# This requires all paths from the previous cell to exist.

RETRIEVAL_LIMIT = 400
EVAL_LIMIT = 100
PILOT_OUTPUT_DIR = Path('/kaggle/working/rgca_pilot_500') if IS_KAGGLE else PROJECT_ROOT / 'outputs' / 'rgca_pilot_500'
BUILT_SUBSET_PATH = PILOT_OUTPUT_DIR / 'data' / 'mimic_subset.jsonl'

if SUBSET_PATH is not None:
    print('Subset already selected, skipping build:', SUBSET_PATH)
else:
    required = {
        'metadata': METADATA_PATH,
        'split': SPLIT_PATH,
        'labels': LABELS_PATH,
        'reports_root': MIMIC_REPORTS_ROOT,
        'images_root': MIMIC_JPG_ROOT,
    }
    missing = {name: str(path) for name, path in required.items() if not path or not Path(path).exists()}
    if missing:
        raise FileNotFoundError('Cannot build subset because files are missing: ' + json.dumps(missing, indent=2))

    run_command([
        sys.executable,
        'scripts/kaggle_run_pilot.py',
        '--metadata', str(METADATA_PATH),
        '--split', str(SPLIT_PATH),
        '--labels', str(LABELS_PATH),
        '--reports-root', str(MIMIC_REPORTS_ROOT),
        '--images-root', str(MIMIC_JPG_ROOT),
        '--output-dir', str(PILOT_OUTPUT_DIR),
        '--limit', str(RETRIEVAL_LIMIT + EVAL_LIMIT),
        '--retrieval-limit', str(RETRIEVAL_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--retriever', 'lexical',
        '--generator', 'mock',
        '--top-k', '3',
    ])
    SUBSET_PATH = BUILT_SUBSET_PATH
    print('Built subset:', SUBSET_PATH)


## 5. Validate Subset

This must show non-zero retrieval and eval records before experiments run.


In [ ]:
if SUBSET_PATH is None or not Path(SUBSET_PATH).exists():
    raise FileNotFoundError('No mimic_subset.jsonl is available. Attach a private Kaggle dataset or build from local PhysioNet files first.')

studies = load_studies(SUBSET_PATH)
retrieval_pool = [study for study in studies if study.split == 'retrieval_pool']
eval_studies = [study for study in studies if study.split == 'eval']

print('SUBSET_PATH:', SUBSET_PATH)
print('total:', len(studies))
print('retrieval_pool:', len(retrieval_pool))
print('eval:', len(eval_studies))
print('example:', studies[0].to_dict() if studies else None)

assert retrieval_pool, 'No retrieval_pool records found.'
assert eval_studies, 'No eval records found.'


## 6. Run Structured Experiment Suite

This runs the planned baseline matrix from config.


In [ ]:
SUITE_OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/mimic_pilot_baseline_v0') if IS_KAGGLE else PROJECT_ROOT / 'outputs' / 'rgca_experiments' / 'notebook_no_gcloud_suite'
SUITE_CONFIG = PROJECT_ROOT / 'configs' / 'mimic_pilot_suite.json'

run_command([
    sys.executable,
    'scripts/run_experiment_suite.py',
    '--config', str(SUITE_CONFIG),
    '--input', str(SUBSET_PATH),
    '--output-dir', str(SUITE_OUTPUT_DIR),
    '--overwrite',
])


## 7. Summarize Results


In [ ]:
TABLES_DIR = SUITE_OUTPUT_DIR / 'tables'
SUITE_MANIFEST = SUITE_OUTPUT_DIR / 'suite_manifest.json'

run_command([
    sys.executable,
    'scripts/summarize_experiment_suite.py',
    '--manifest', str(SUITE_MANIFEST),
    '--output-dir', str(TABLES_DIR),
])

print((TABLES_DIR / 'suite_summary.md').read_text(encoding='utf-8')[:6000])


## 8. Inspect Mismatch Examples


In [ ]:
EXPERIMENT_TO_INSPECT = 'E02_stress_lexical_k3'
DETAILS_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'evaluation' / 'mismatch' / 'evaluation_details.jsonl'
GENERATIONS_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'baseline' / 'generations_mismatch.jsonl'
RETRIEVAL_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'baseline' / 'mismatch_results.jsonl'

details = read_jsonl(DETAILS_PATH)
generations = {row['study_id']: row for row in read_jsonl(GENERATIONS_PATH)}
retrievals = {row['target_study']: row for row in read_jsonl(RETRIEVAL_PATH)}
interesting = [row for row in details if row['retrieval_induced_flags']]

print('cases_with_retrieval_induced_flags:', len(interesting))
for row in interesting[:5]:
    study_id = row['study_id']
    print('
' + '=' * 100)
    print('study_id:', study_id)
    print('reference_labels:', row['reference_labels'])
    print('retrieved_labels:', row['retrieved_labels'])
    print('generated_labels:', row['generated_labels'])
    print('retrieval_induced_flags:', row['retrieval_induced_flags'])
    print('
Generated report snippet:')
    print(generations[study_id]['generated_report'][:1200])
    print('
First retrieved report snippet:')
    print(retrievals[study_id]['retrieved_reports'][0][:1200])


## 9. Package Private Kaggle Dataset For Future Runs

This creates `/kaggle/working/rgca_private_dataset.zip` containing `mimic_subset.jsonl` and experiment outputs. Upload it as a **Private** Kaggle dataset, then future notebooks can attach it as input and skip all data restore/build steps.


In [ ]:
PRIVATE_DATASET_DIR = Path('/kaggle/working/rgca_private_dataset') if IS_KAGGLE else PROJECT_ROOT / 'outputs' / 'rgca_private_dataset'

run_command([
    sys.executable,
    'scripts/package_private_kaggle_dataset.py',
    '--subset-jsonl', str(SUBSET_PATH),
    '--output-dir', str(PRIVATE_DATASET_DIR),
    '--suite-output-dir', str(SUITE_OUTPUT_DIR),
    '--zip',
])

zip_path = PRIVATE_DATASET_DIR.with_suffix('.zip')
print('Private dataset folder:', PRIVATE_DATASET_DIR)
print('Private dataset zip:', zip_path)
print('Zip exists:', zip_path.exists())
if zip_path.exists():
    print('Size MB:', round(zip_path.stat().st_size / (1024 * 1024), 2))


## 10. Save As Private Kaggle Dataset

In Kaggle:

1. Open the right-side Output panel.
2. Download or locate `rgca_private_dataset.zip`.
3. Go to Kaggle Datasets -> New Dataset.
4. Upload the zip contents.
5. Set visibility to **Private**.
6. In future notebooks, click **Add Input** and attach that private dataset.

Keep it private because it may contain MIMIC-derived report text.
